# BlogPulse: users in PostgreSQL, posts in MongoDB
**DBMS for Analytics (25MSDA1S04) · Problem 19**

This notebook connects to both databases from Python, runs the analytical queries and aggregation pipelines, and charts the results.

| Part | Database | Driver | Data |
|---|---|---|---|
| 1 | PostgreSQL | `psycopg2` (Python DB-API 2.0) | users, roles, plans, subscriptions, logins, audit log |
| 2 | MongoDB | `pymongo` | posts with embedded comments and tags |
| 3 | Both | pandas | join MongoDB post stats with SQL user attributes |

**Before running:** create the database and load `schema.sql` and `data.sql` (see README), and load `mongo/posts.json` into MongoDB (the Mongo section does this automatically if the collection is empty). Connection settings are read from environment variables.

In [ ]:
import os, json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import psycopg2                      # PostgreSQL DB-API driver

os.makedirs("charts", exist_ok=True)

# Brand colours used in every chart (one hue per job; grey for context)
BLUE, GREEN, GREY = "#0062A9", "#07963A", "#6B7280"

def style(ax, title, xlabel="", ylabel=""):
    ax.set_title(title, loc="left", fontsize=12, fontweight="bold", color="#1F2937", pad=12)
    ax.set_xlabel(xlabel, color=GREY); ax.set_ylabel(ylabel, color=GREY)
    for side in ("top", "right"): ax.spines[side].set_visible(False)
    for side in ("left", "bottom"): ax.spines[side].set_color("#C9D2DC")
    ax.grid(axis="y", color="#E5E7EB", linewidth=0.8); ax.set_axisbelow(True)
    ax.tick_params(colors=GREY, length=0)

def month_axis(ax):
    ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 7]))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))

def save(fig, name):
    fig.tight_layout(); fig.savefig(f"charts/{name}.png", dpi=160); plt.show()

## Part 1 · PostgreSQL through the DB-API

In [ ]:
conn = psycopg2.connect(
    host=os.getenv("PGHOST", "localhost"),
    port=os.getenv("PGPORT", "5432"),
    user=os.getenv("PGUSER", "postgres"),
    password=os.getenv("PGPASSWORD", ""),
    dbname=os.getenv("PGDATABASE", "blogpulse"),
)

def query_df(sql, params=None):
    """Run a query through a DB-API cursor and return a pandas DataFrame."""
    with conn.cursor() as cur:
        cur.execute(sql, params)
        cols = [d[0] for d in cur.description]
        return pd.DataFrame(cur.fetchall(), columns=cols)

# Row counts: the project needs at least 500 rows across tables
tables = ["roles", "countries", "plans", "users", "subscriptions", "user_logins", "account_audit_log"]
counts = pd.DataFrame(
    [(t, query_df(f"SELECT COUNT(*) AS n FROM {t}")["n"][0]) for t in tables],
    columns=["table", "rows"])
print(counts.to_string(index=False)); print("TOTAL ROWS:", counts["rows"].sum())

### 1.1 Sign-ups by month (view `v_monthly_signups`)

In [ ]:
signups = query_df("SELECT month, signups, cumulative_users::int AS cumulative FROM v_monthly_signups ORDER BY month")
signups["month"] = pd.to_datetime(signups["month"])

fig, ax = plt.subplots(figsize=(7, 3.6))
ax.bar(signups["month"], signups["signups"], width=20, color=BLUE)
peak = signups.loc[signups["signups"].idxmax()]
ax.annotate(f"Peak: {int(peak.signups)} sign-ups", (peak.month, peak.signups),
            xytext=(0, 8), textcoords="offset points", ha="center", color="#1F2937", fontsize=9)
style(ax, f"Sign-ups per month ({int(signups['cumulative'].iloc[-1])} users in total)", ylabel="New users")
ax.yaxis.get_major_locator().set_params(integer=True); month_axis(ax)
save(fig, "signups_by_month")

### 1.2 Monthly active users (distinct users who logged in)

In [ ]:
mau = query_df("""
    SELECT DATE_TRUNC('month', login_at)::date AS month,
           COUNT(DISTINCT user_id)::int        AS active_users,
           COUNT(*)::int                       AS logins
    FROM user_logins GROUP BY 1 ORDER BY 1""")
mau["month"] = pd.to_datetime(mau["month"])

fig, ax = plt.subplots(figsize=(5, 3.6))
ax.plot(mau["month"], mau["active_users"], color=BLUE, linewidth=2)
last = mau.iloc[-1]
ax.plot(last.month, last.active_users, "o", color=BLUE, markersize=7, markeredgecolor="white", markeredgewidth=2)
ax.annotate(f"{int(last.active_users)} active users", (last.month, last.active_users),
            xytext=(-8, 8), textcoords="offset points", ha="right", color="#1F2937", fontsize=9)
style(ax, "Monthly active users", ylabel="Users with at least one login"); month_axis(ax)
save(fig, "monthly_active_users")

### 1.3 Role distribution

In [ ]:
roles = query_df("""
    SELECT r.role_name, COUNT(*)::int AS users
    FROM users u JOIN roles r ON r.role_id = u.role_id
    GROUP BY r.role_name ORDER BY users""")
total = roles["users"].sum()

fig, ax = plt.subplots(figsize=(7, 3.2))
bars = ax.barh(roles["role_name"], roles["users"], color=BLUE, height=0.55)
for b, n in zip(bars, roles["users"]):
    ax.text(b.get_width() + 2, b.get_y() + b.get_height()/2, f"{n}  ({n/total:.0%})", va="center", color="#1F2937", fontsize=9)
style(ax, "Users by role", xlabel="Users"); ax.grid(axis="y", visible=False); ax.grid(axis="x", color="#E5E7EB")
ax.set_xlim(0, roles["users"].max() * 1.2)
save(fig, "role_distribution")

### 1.4 Revenue by plan (active subscriptions)

In [ ]:
mrr = query_df("""
    SELECT p.plan_name, COUNT(*)::int AS subscribers, SUM(p.monthly_price)::float AS mrr
    FROM subscriptions s JOIN plans p ON p.plan_id = s.plan_id
    WHERE s.status = 'active' GROUP BY p.plan_name ORDER BY mrr DESC""")
print(mrr.to_string(index=False), "\nTotal MRR:", mrr["mrr"].sum())

fig, ax = plt.subplots(figsize=(5, 3.6))
bars = ax.bar(mrr["plan_name"], mrr["mrr"], color=BLUE, width=0.5)
for b, m, s in zip(bars, mrr["mrr"], mrr["subscribers"]):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 15, f"{m:,.0f}\n{s} subscribers", ha="center", color="#1F2937", fontsize=9)
style(ax, f"Monthly recurring revenue by plan (total {mrr['mrr'].sum():,.0f})", ylabel="MRR (USD)")
ax.set_ylim(0, mrr["mrr"].max() * 1.25)
save(fig, "mrr_by_plan")

### 1.5 When do people log in? (hour of day by device)

In [ ]:
hours = query_df("""
    SELECT EXTRACT(HOUR FROM login_at)::int AS hour,
           COUNT(*) FILTER (WHERE device='web')::int     AS web,
           COUNT(*) FILTER (WHERE device='android')::int AS android,
           COUNT(*) FILTER (WHERE device='ios')::int     AS ios
    FROM user_logins GROUP BY 1 ORDER BY 1""")
hours["total"] = hours[["web", "android", "ios"]].sum(axis=1)

fig, ax = plt.subplots(figsize=(7, 3.6))
ax.bar(hours["hour"], hours["total"], color=BLUE, width=0.7)
top = hours.nlargest(1, "total").iloc[0]
ax.annotate(f"Busiest hour: {int(top.hour)}:00", (top.hour, top.total), xytext=(0, 6),
            textcoords="offset points", ha="center", fontsize=9, color="#1F2937")
style(ax, "Logins by hour of day", xlabel="Hour (0-23)", ylabel="Logins"); ax.set_xticks(range(0, 24, 2))
save(fig, "logins_by_hour")
device_share = (hours[["web", "android", "ios"]].sum() / hours["total"].sum()).round(3)
print("Device share:", device_share.to_dict())

### 1.6 Upgrade conversion and dormant accounts

In [ ]:
conv = query_df("""
    WITH first_sub AS (SELECT DISTINCT ON (user_id) user_id, plan_id FROM subscriptions ORDER BY user_id, start_date)
    SELECT COUNT(*)::int AS started_free,
           COUNT(*) FILTER (WHERE EXISTS (SELECT 1 FROM subscriptions s JOIN plans p ON p.plan_id=s.plan_id
                                          WHERE s.user_id=f.user_id AND p.monthly_price>0))::int AS later_paid
    FROM first_sub f WHERE f.plan_id = 1""")
print(conv.assign(conversion_pct=(100*conv.later_paid/conv.started_free).round(1)).to_string(index=False))

dormant = query_df("""
    SELECT COUNT(*)::int AS dormant_users FROM users u
    WHERE u.is_active AND NOT EXISTS (
        SELECT 1 FROM user_logins l WHERE l.user_id = u.user_id
          AND l.login_at > (SELECT MAX(login_at) FROM user_logins) - INTERVAL '90 days')""")
print(dormant.to_string(index=False))

### What the SQL data shows
* **Growth is real, not just sign-ups.** Monthly sign-ups roughly tripled between early 2024 and 2025, and monthly active users grew from single digits to ~90.
* **Revenue is concentrated.** The Team plan has the fewest paying subscribers but brings in most of the recurring revenue.
* **Usage has evening and morning peaks**, which tells editors when to schedule publishing.

## Part 2 · MongoDB with PyMongo

In [ ]:
# [mongo]
from pymongo import MongoClient
from bson import json_util

MONGO_URI = os.getenv("MONGO_URI", "mongodb://localhost:27017")
mdb = MongoClient(MONGO_URI, serverSelectionTimeoutMS=4000)[os.getenv("MONGO_DB", "blogpulse")]
posts = mdb.posts

# Load the sample posts if the collection is empty, then create the indexes
if posts.count_documents({}) == 0:
    posts.insert_many(json_util.loads(Path("mongo/posts.json").read_text(encoding="utf-8")))
posts.create_index("slug", unique=True)
posts.create_index("tags")
posts.create_index([("author_id", 1), ("created_at", -1)])
posts.create_index([("title", "text"), ("body", "text")])

print("posts:", posts.count_documents({}),
      "| embedded comments:", next(posts.aggregate([{"$group": {"_id": None, "n": {"$sum": {"$size": "$comments"}}}}]))["n"])
print("indexes:", [i["name"] for i in posts.list_indexes()])

### 2.1 CRUD (insert · find · update · delete)

In [ ]:
# [mongo]
doc = {"title": "Notebook demo post", "slug": "notebook-demo-post", "author_id": 2, "author_name": "Liam Reddy",
       "category": "Tutorial", "tags": ["mongodb", "python"], "body": "Demo", "status": "published",
       "views": 0, "likes": 0, "comments": []}
from datetime import datetime, timezone
doc["created_at"] = datetime.now(timezone.utc)

posts.delete_many({"slug": doc["slug"]})                                  # keep the cell re-runnable
posts.insert_one(doc)                                                      # CREATE
print("found:", posts.find_one({"slug": doc["slug"]}, {"title": 1, "_id": 0}))   # READ
posts.update_one({"slug": doc["slug"]}, {"$push": {"comments": {"comment_id": 1, "user_id": 10,
                 "user_name": "Demo Reader", "text": "Nice!", "likes": 0, "created_at": datetime.now(timezone.utc)}},
                 "$inc": {"views": 1}})                                    # UPDATE (embedded comment)
print("after update:", posts.find_one({"slug": doc["slug"]}, {"views": 1, "comments.text": 1, "_id": 0}))
print("deleted:", posts.delete_one({"slug": doc["slug"]}).deleted_count)  # DELETE

### 2.2 Aggregation pipelines

In [ ]:
# [mongo]
def agg(pipeline):
    return pd.DataFrame(list(posts.aggregate(pipeline)))

top_posts = agg([
    {"$match": {"status": "published"}},
    {"$addFields": {"comment_count": {"$size": "$comments"}}},
    {"$addFields": {"engagement": {"$add": ["$views", {"$multiply": ["$likes", 5]}, {"$multiply": ["$comment_count", 10]}]}}},
    {"$sort": {"engagement": -1}}, {"$limit": 10},
    {"$project": {"_id": 0, "title": 1, "author_name": 1, "views": 1, "likes": 1, "comment_count": 1, "engagement": 1}},
])
popular_tags = agg([
    {"$match": {"status": "published"}},
    {"$unwind": "$tags"},
    {"$group": {"_id": "$tags", "posts": {"$sum": 1}, "avg_views": {"$avg": "$views"},
                "total_comments": {"$sum": {"$size": "$comments"}}}},
    {"$sort": {"posts": -1, "_id": 1}}, {"$limit": 10},
    {"$project": {"_id": 0, "tag": "$_id", "posts": 1, "avg_views": {"$round": ["$avg_views", 0]}, "total_comments": 1}},
])
by_author = agg([
    {"$match": {"status": "published"}},
    {"$group": {"_id": {"id": "$author_id", "name": "$author_name"}, "posts": {"$sum": 1},
                "comments": {"$sum": {"$size": "$comments"}}, "views": {"$sum": "$views"}}},
    {"$addFields": {"comments_per_post": {"$round": [{"$divide": ["$comments", "$posts"]}, 2]}}},
    {"$sort": {"comments": -1}}, {"$limit": 10},
    {"$project": {"_id": 0, "author_id": "$_id.id", "author": "$_id.name", "posts": 1, "comments": 1,
                  "comments_per_post": 1, "views": 1}},
])
display(top_posts); display(popular_tags); display(by_author)

In [ ]:
# [mongo]
fig, ax = plt.subplots(figsize=(7, 3.8))
d = popular_tags.sort_values("posts")
bars = ax.barh(d["tag"], d["posts"], color=BLUE, height=0.6)
for b, n in zip(bars, d["posts"]): ax.text(b.get_width() + 0.3, b.get_y() + b.get_height()/2, int(n), va="center", fontsize=9, color="#1F2937")
style(ax, "Most used tags (published posts)", xlabel="Posts"); ax.grid(axis="y", visible=False); ax.grid(axis="x", color="#E5E7EB")
save(fig, "top_tags")

fig, ax = plt.subplots(figsize=(7, 3.8))
d = by_author.sort_values("comments")
bars = ax.barh(d["author"], d["comments"], color=GREEN, height=0.6)
for b, n in zip(bars, d["comments"]): ax.text(b.get_width() + 0.15, b.get_y() + b.get_height()/2, int(n), va="center", fontsize=9, color="#1F2937")
style(ax, "Comments received per author (top 10)", xlabel="Comments"); ax.grid(axis="y", visible=False); ax.grid(axis="x", color="#E5E7EB")
save(fig, "comments_per_author")

## Part 3 · Using both databases together
Post statistics live in MongoDB, account attributes in PostgreSQL. The shared key is `author_id` = `users.user_id`, so pandas can join the two results.

In [ ]:
# [mongo]
author_stats = agg([
    {"$match": {"status": "published"}},
    {"$group": {"_id": "$author_id", "posts": {"$sum": 1}, "views": {"$sum": "$views"},
                "comments": {"$sum": {"$size": "$comments"}}}},
    {"$project": {"_id": 0, "user_id": "$_id", "posts": 1, "views": 1, "comments": 1}},
])
sql_users = query_df("SELECT user_id, username, role_name, country_name, current_plan FROM v_user_overview")
joined = author_stats.merge(sql_users, on="user_id", how="left")

by_role = joined.groupby("role_name")[["posts", "views", "comments"]].sum().sort_values("views", ascending=False)
by_plan = joined.groupby("current_plan").agg(authors=("user_id", "count"), views_per_author=("views", "mean")).round(0)
display(by_role); display(by_plan)

## SQL vs NoSQL: why each part lives where it does

| Aspect | PostgreSQL (users, roles, plans) | MongoDB (posts, comments, tags) |
|---|---|---|
| Schema | Fixed, enforced with types and CHECK constraints | Flexible: a post may have `cover_image`, `series` or `reading_time_min`, another may not |
| Relationships | Foreign keys and JOINs; comments would need a separate table plus JOIN | Comments and tags are embedded, so one read returns the whole post |
| Integrity | Strong: unique e-mail, one active subscription, triggers, ACID across tables | Per-document atomicity; integrity of `author_id` is checked by the application |
| Best for | Accounts, money, audit trails | Content that changes shape and is read as a whole |
| Analytics style | GROUP BY, window functions, views | Pipelines with `$unwind`, `$group`, `$project` |
| Our choice | Users, roles, subscriptions, logins, audit | Posts with embedded comments and tag arrays |

**Why posts suit a document model:** a post is read together with its comments and tags, and its structure keeps evolving. Embedding avoids joins on every page view and lets new optional fields appear without a schema migration.

In [ ]:
conn.close()